# E1: F3Loc baseline reproduction on Gibson (Student A)

Reproduces F3Loc (Chen et al., CVPR 2024) Tables 1, 2 and 3 with the **released checkpoints, no training**, on the Gibson Floorplan Localization Dataset.

**Data source:** the dataset zips you uploaded to your Google Drive folder `Thesis` (`desdf.zip`, `gibson_t.zip`, `gibson_f.zip`, `gibson_g.zip`).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all. Approve both Google sign-in pop-ups (Drive mount and Drive access).

**How the data is handled.** Colab's disk (~75 GB) cannot hold all subsets at once, and `gibson_g` alone is ~55 GB. So the notebook handles **one subset at a time**: it streams the zip from Drive and unzips it on the fly (the zip itself is never copied to Colab), runs that subset's evaluations, deletes the unzipped files, then moves to the next subset.

**Resumable.** Every result is written to Drive as soon as it finishes. If Colab disconnects, Run all again: finished evaluations are skipped, and a subset whose evaluations are all done is not even unzipped.

**Timing caveat.** Colab GPUs are shared VMs. The timing table is context only, not a lab measurement.

## 1. Configuration (the only cell you should edit)

In [1]:
RUN_ID = "gibson_v1"                       # results folder name; rerunning with the same ID resumes
DATA_FOLDER = "/content/drive/MyDrive/Thesis"           # where the dataset zips are
DATA_FOLDER_ID = "19m6fFCnH0Ago3Xs9HPu4Yn1Cw2xEJVCQ"    # the same folder's Drive ID (from its link)
DRIVE_E1 = "/content/drive/MyDrive/Thesis/E1"            # results and checkpoint cache go here
SUBSETS = ["gibson_t", "gibson_f", "gibson_g"]          # processed in this order; remove one to skip it
DELETE_AFTER = True                        # delete each unzipped subset after its evaluations (needed for disk space)
F3LOC_COMMIT = "9e8027d9219ca505078283ebfd925580f476ab97"
CKPT_FOLDER_URL = "https://drive.google.com/drive/folders/1-TDlM9hjeODizeebgfPx7zWez0XPYCKm"
TRAJ_LEN = 100                             # filter history used for the paper's Table 3
PINS = {"lightning": "2.5.6", "gdown": "6.4.2"}

## 2. GPU and Drive

In [2]:
import os, torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Run all again."
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
RESULTS = f"{DRIVE_E1}/results/{RUN_ID}"
CACHE = f"{DRIVE_E1}/cache"
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(f"{CACHE}/ckpt", exist_ok=True)
print("results ->", RESULTS)
print("zips in", DATA_FOLDER, ":", sorted(f for f in os.listdir(DATA_FOLDER) if f.endswith(".zip")))

GPU: Tesla T4
Mounted at /content/drive
results -> /content/drive/MyDrive/Thesis/E1/results/gibson_v1
zips in /content/drive/MyDrive/Thesis : ['desdf.zip', 'gibson_f.zip', 'gibson_g.zip', 'gibson_t.zip']


## 3. Install pinned packages

In [3]:
!pip install -q lightning=={PINS['lightning']} gdown=={PINS['gdown']}

## 4. Embedded files
The logging patch and the comparison script. Identical to `f3loc-e1-logging.patch` and `compare_e1.py`.

In [4]:
%%writefile /content/f3loc-e1-logging.patch
diff --git a/eval_filtering.py b/eval_filtering.py
index b034696..0e9ea49 100644
--- a/eval_filtering.py
+++ b/eval_filtering.py
@@ -1,4 +1,5 @@
 import argparse
+import csv
 import os
 import time

@@ -6,7 +7,11 @@ import matplotlib.pyplot as plt
 import torch
 import tqdm
 import yaml
-from attrdict import AttrDict
+# [E1] attrdict 2.0.1 fails to import on Python >= 3.10 (collections.Mapping removed).
+# The scripts only read split.test, so a two-line shim is equivalent.
+class AttrDict(dict):
+    __getattr__ = dict.__getitem__
+
 from torch.utils.data import DataLoader

 from modules.comp.comp_d_net_pl import *
@@ -48,6 +53,10 @@ def evaluate_filtering():
     parser.add_argument(
         "--traj_len", type=int, default=100, help="length of the trajectory"
     )
+    # [E1] per-trajectory and per-step output; does not change the algorithm
+    parser.add_argument(
+        "--out_prefix", type=str, default=None, help="[E1] prefix for CSV outputs"
+    )
     args = parser.parse_args()

     # get device
@@ -190,6 +199,8 @@ def evaluate_filtering():
     iteration_time = 0
     feature_extraction_time = 0
     n_iter = 0
+    traj_rows = []  # [E1]
+    step_rows = []  # [E1] per-step timings, so we can report p50/p99/max not a mean

     # loop the over scenes
     for data_idx in tqdm.tqdm(range(len(test_set))):
@@ -422,6 +433,15 @@ def evaluate_filtering():
             )

             end_iter = time.time()
+            step_rows.append(  # [E1]
+                dict(
+                    data_idx=data_idx,
+                    t=t,
+                    feature_s=feature_extraction_end - feature_extraction_start,
+                    matching_s=matching_end - matching_start,
+                    iteration_s=end_iter - start_iter,
+                )
+            )
             matching_time += matching_end - matching_start
             feature_extraction_time += feature_extraction_end - feature_extraction_start
             iteration_time += end_iter - start_iter
@@ -441,6 +461,14 @@ def evaluate_filtering():
                 .mean()
             ) ** 0.5 * 0.01
             RMSEs.append(RMSE)
+            traj_rows.append(  # [E1]
+                dict(
+                    data_idx=data_idx,
+                    scene=scene,
+                    rmse_m=float(RMSE),
+                    max_last10_err_m=float(last_errors.max()),
+                )
+            )
             print("last_errors", last_errors)
             if all(last_errors < 1):
                 success_10.append(True)
@@ -477,6 +505,18 @@ def evaluate_filtering():
         print("mean RMSE succeeded : ", RMSEs[success_10].mean())
         print("mean RMSE all : ", RMSEs.mean())

+    # [E1] guard against silently dropped trajectories, then dump CSVs
+    print("trajectories evaluated = ", len(RMSEs), " / dataset length = ", len(test_set))
+    assert len(RMSEs) == len(test_set), "trajectories were dropped"
+    if args.out_prefix is not None:
+        os.makedirs(os.path.dirname(os.path.abspath(args.out_prefix)), exist_ok=True)
+        for suffix, rows in (("traj", traj_rows), ("steps", step_rows)):
+            with open(f"{args.out_prefix}_{suffix}.csv", "w", newline="") as f:
+                f.write(f"# net_type={net_type} traj_len={traj_l} device={device}\n")
+                w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
+                w.writeheader()
+                w.writerows(rows)
+
     if log_timing:
         feature_extraction_time = feature_extraction_time / n_iter
         matching_time = matching_time / n_iter
diff --git a/eval_observation.py b/eval_observation.py
index 58cf79e..336ea12 100644
--- a/eval_observation.py
+++ b/eval_observation.py
@@ -1,10 +1,15 @@
 import argparse
+import csv
 import os

 import torch
 import tqdm
 import yaml
-from attrdict import AttrDict
+# [E1] attrdict 2.0.1 fails to import on Python >= 3.10 (collections.Mapping removed).
+# The scripts only read split.test, so a two-line shim is equivalent.
+class AttrDict(dict):
+    __getattr__ = dict.__getitem__
+

 from modules.comp.comp_d_net_pl import *
 from modules.mono.depth_net_pl import *
@@ -43,6 +48,10 @@ def evaluate_observation():
     parser.add_argument(
         "--ckpt_path", type=str, default="./logs", help="path of the checkpoints"
     )
+    # [E1] per-frame output; does not change the algorithm
+    parser.add_argument(
+        "--out_csv", type=str, default=None, help="[E1] write one row per frame"
+    )
     args = parser.parse_args()

     # get device
@@ -168,6 +177,7 @@ def evaluate_observation():
     # record the accuracy
     acc_record = []
     acc_orn_record = []
+    rows = []  # [E1]
     for data_idx in tqdm.tqdm(range(len(test_set))):
         data = test_set[data_idx]
         # get the scene name according to the data_idx
@@ -258,6 +268,32 @@ def evaluate_observation():
         acc_orn = (pose_pred[2] - gt_pose_desdf[2]) % (2 * np.pi)
         acc_orn = min(acc_orn, 2 * np.pi - acc_orn) / np.pi * 180
         acc_orn_record.append(acc_orn)
+        rows.append(  # [E1]
+            dict(
+                data_idx=data_idx,
+                scene=scene,
+                idx_within_scene=int(idx_within_scene),
+                pred_x=float(pose_pred[0]),
+                pred_y=float(pose_pred[1]),
+                pred_yaw=float(pose_pred[2]),
+                gt_x=float(gt_pose_desdf[0]),
+                gt_y=float(gt_pose_desdf[1]),
+                gt_yaw=float(gt_pose_desdf[2]),
+                err_m=float(acc),
+                err_deg=float(acc_orn),
+            )
+        )
+
+    # [E1] guard against silently dropped frames
+    print("frames evaluated = ", len(acc_record), " / dataset length = ", len(test_set))
+    assert len(acc_record) == len(test_set), "frames were dropped"
+    if args.out_csv is not None:
+        os.makedirs(os.path.dirname(os.path.abspath(args.out_csv)), exist_ok=True)
+        with open(args.out_csv, "w", newline="") as f:
+            f.write(f"# net_type={net_type} dataset={args.dataset} device={device}\n")
+            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
+            w.writeheader()
+            w.writerows(rows)

     acc_record = np.array(acc_record)
     acc_orn_record = np.array(acc_orn_record)

Writing /content/f3loc-e1-logging.patch


In [5]:
%%writefile /content/compare_e1.py
"""E1: build the ours-vs-paper table from the per-frame CSVs written by the patched eval scripts.

Usage: python compare_e1.py <results_dir>

Paper numbers are from F3Loc (Chen et al., CVPR 2024), arXiv 2403.03370v2, Tables 1-4.
Row mapping (net_type in the repo -> name in the paper):
    d      -> Ours_s  (monocular)
    mvd    -> Ours_m  (multi-view)
    comp_s -> Ours_t  (hard-threshold selection)
    comp   -> Ours_f  (learned fusion)
Verdict per the research plan: |delta| <= 1 pt OK, 1-2 pts CHECK, > 2 pts STOP and find out why.
"""
import csv
import math
import os
import sys

import numpy as np

PAPER_OBS = {  # (dataset, net) -> R@0.1m, R@0.5m, R@1m, R@1m30deg, in percent
    ("gibson_f", "d"): (4.7, 28.6, 36.6, 35.1),
    ("gibson_f", "mvd"): (13.2, 40.9, 45.2, 43.7),
    ("gibson_g", "d"): (4.3, 26.7, 33.7, 32.3),
    ("gibson_g", "mvd"): (9.3, 27.0, 31.0, 29.2),
    ("gibson_g", "comp_s"): (10.5, 34.3, 39.6, 38.0),
    ("gibson_g", "comp"): (12.2, 39.4, 44.5, 43.2),
}
PAPER_FILT = {  # net -> success@1m (%), RMSE succeeded (m), RMSE all (m)
    "d": (89.2, 0.18, 0.88),
    "comp": (94.6, 0.12, 0.51),
}
PAPER_TIMING_MEAN_S = {"comp": (0.033, 0.003, 0.037)}  # feature, matching, iteration; hardware unstated
NAMES = {"d": "Ours_s", "mvd": "Ours_m", "comp_s": "Ours_t", "comp": "Ours_f"}


def read_csv(path):
    with open(path) as f:
        meta = f.readline().strip()
        rows = list(csv.DictReader(f))
    return meta, rows


def verdict(delta_pts):
    a = abs(delta_pts)
    return "OK" if a <= 1 else ("CHECK" if a <= 2 else "STOP")


def ci95(p, n):
    """95% normal-approx half-width for a proportion p in [0,1], in percentage points."""
    return 196 * math.sqrt(max(p * (1 - p), 1e-12) / n)


def obs_section(out):
    lines = [
        "## Single-frame observation (Tables 1 and 2)",
        "",
        "| dataset | net | paper row | N | metric | ours (%) | ±95% | paper (%) | delta | verdict |",
        "|---|---|---|---|---|---|---|---|---|---|",
    ]
    for ds in ("gibson_f", "gibson_g"):
        for net in ("d", "mvd", "comp_s", "comp"):
            path = os.path.join(out, f"obs_{ds}_{net}.csv")
            if not os.path.exists(path):
                continue
            _, rows = read_csv(path)
            e = np.array([float(r["err_m"]) for r in rows])
            o = np.array([float(r["err_deg"]) for r in rows])
            n = len(e)
            ours = (
                np.mean(e < 0.1),
                np.mean(e < 0.5),
                np.mean(e < 1.0),
                np.mean((e < 1.0) & (o < 30)),
            )
            paper = PAPER_OBS.get((ds, net))
            for k, m in enumerate(("R@0.1m", "R@0.5m", "R@1m", "R@1m30°")):
                ours_pct = 100 * ours[k]
                if paper is None:
                    p, d, v = "not reported", "", ""
                else:
                    p = f"{paper[k]:.1f}"
                    d = f"{ours_pct - paper[k]:+.1f}"
                    v = verdict(ours_pct - paper[k])
                lines.append(
                    f"| {ds} | {net} | {NAMES[net]} | {n} | {m} | {ours_pct:.1f} | "
                    f"{ci95(ours[k], n):.1f} | {p} | {d} | {v} |"
                )
    return lines


def filt_section(out):
    lines = [
        "",
        "## Sequential filtering on gibson_t, 100-frame history (Table 3)",
        "",
        "| net | paper row | trajectories | success@1m ours (%) | paper | delta | RMSE succ ours / paper (m) | RMSE all ours / paper (m) | verdict |",
        "|---|---|---|---|---|---|---|---|---|",
    ]
    timing = [
        "",
        "## Per-step timing (Table 4 context; ours on our GPU, paper hardware unstated)",
        "",
        "Lab rule: distributions only. The paper's column is its reported mean, shown for orientation, not compared.",
        "",
        "| net | stage | N steps | p50 (ms) | p99 (ms) | p99.9 (ms) | max (ms) | paper mean (ms) |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for net in ("d", "comp"):
        tpath = os.path.join(out, f"filt_{net}_traj.csv")
        if os.path.exists(tpath):
            _, rows = read_csv(tpath)
            rm = np.array([float(r["rmse_m"]) for r in rows])
            mx = np.array([float(r["max_last10_err_m"]) for r in rows])
            succ = mx < 1.0
            sr = 100 * succ.mean()
            p = PAPER_FILT[net]
            rs = rm[succ].mean() if succ.any() else float("nan")
            lines.append(
                f"| {net} | {NAMES[net]} | {len(rm)} | {sr:.1f} | {p[0]:.1f} | {sr - p[0]:+.1f} | "
                f"{rs:.2f} / {p[1]:.2f} | {rm.mean():.2f} / {p[2]:.2f} | {verdict(sr - p[0])} |"
            )
        spath = os.path.join(out, f"filt_{net}_steps.csv")
        if os.path.exists(spath):
            _, rows = read_csv(spath)
            for k, (col, label) in enumerate(
                (("feature_s", "feature"), ("matching_s", "matching"), ("iteration_s", "iteration"))
            ):
                x = 1000 * np.array([float(r[col]) for r in rows])
                pm = PAPER_TIMING_MEAN_S.get(net)
                pm = f"{1000 * pm[k]:.0f}" if pm else "n/a"
                q = np.percentile(x, [50, 99, 99.9])
                timing.append(
                    f"| {net} | {label} | {len(x)} | {q[0]:.2f} | {q[1]:.2f} | {q[2]:.2f} | {x.max():.2f} | {pm} |"
                )
    return lines + timing


def main():
    out = sys.argv[1]
    lines = ["# E1 results: F3Loc released checkpoints vs. paper", ""]
    prov = os.path.join(out, "provenance.txt")
    if os.path.exists(prov):
        lines += ["```", open(prov).read().strip(), "```", ""]
    lines += obs_section(out) + filt_section(out)
    print("\n".join(lines))


if __name__ == "__main__":
    main()

Writing /content/compare_e1.py


## 5. Clone F3Loc at the pinned commit and apply the patch

In [6]:
%cd /content
!rm -rf f3loc && git clone -q https://github.com/felix-ch/f3loc.git && cd f3loc && git checkout -q {F3LOC_COMMIT} && git apply /content/f3loc-e1-logging.patch && git log -1 --format='commit %H (%cd)' && git diff --stat

/content
commit 9e8027d9219ca505078283ebfd925580f476ab97 (Sat Aug 31 00:29:03 2024 +0200)
 eval_filtering.py   | 42 +++++++++++++++++++++++++++++++++++++++++-
 eval_observation.py | 38 +++++++++++++++++++++++++++++++++++++-
 2 files changed, 78 insertions(+), 2 deletions(-)


## 6. Checkpoints
Downloaded once and cached in `Thesis/E1/cache/ckpt/`. If `gdown` hits a Google Drive quota error, download the checkpoint folder in your browser (link in the F3Loc README) and put `mono.ckpt`, `mv.ckpt`, `comp.ckpt` in `Thesis/E1/cache/ckpt/`, then rerun this cell.

In [7]:
import glob, shutil
names = ["mono.ckpt", "mv.ckpt", "comp.ckpt"]
if not all(os.path.exists(f"{CACHE}/ckpt/{n}") for n in names):
    !gdown -q --folder {CKPT_FOLDER_URL} -O /content/ckpt_dl
    found = {os.path.basename(p): p for p in glob.glob("/content/ckpt_dl/**/*.ckpt", recursive=True)}
    missing = [n for n in names if n not in found]
    assert not missing, f"Missing from download: {missing}. Found: {list(found)}. See the note above this cell."
    for n in names:
        shutil.copy(found[n], f"{CACHE}/ckpt/{n}")
os.makedirs("/content/f3loc/logs", exist_ok=True)
for n in names:
    shutil.copy(f"{CACHE}/ckpt/{n}", f"/content/f3loc/logs/{n}")
!ls -la /content/f3loc/logs

total 414028
drwxr-xr-x 2 root root      4096 Oct 25 14:56 .
drwxr-xr-x 7 root root      4096 Oct 25 14:56 ..
-rw------- 1 root root 106202057 Oct 25 14:56 comp.ckpt
-rw------- 1 root root 311672249 Oct 25 14:56 mono.ckpt
-rw------- 1 root root   6070821 Oct 25 14:56 mv.ckpt


## 7. Data access helpers
Opens each zip in Drive as a seekable stream, so a zip can be checked and unzipped without copying it to Colab. Uses the Drive API (one extra sign-in pop-up); if that fails it falls back to reading through the Drive mount.

In [8]:
import io, time, zipfile, requests
from tqdm.auto import tqdm

class RangeFile(io.RawIOBase):
    """Read-only, seekable view of a remote file. Keeps one streaming connection open while
    reads are sequential; reconnects (with retries) on a backwards seek or a network error."""
    def __init__(self, size, fetch, retries=8, skip_ahead=8 << 20):
        self.size, self.fetch, self.retries, self.skip_ahead = size, fetch, retries, skip_ahead
        self.pos, self._it, self._chunk, self._cpos, self.reconnects = 0, None, b"", 0, 0
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=0):
        self.pos = {0: off, 1: self.pos + off, 2: self.size + off}[whence]
        return self.pos
    def _next_chunk(self):
        chunk = next(self._it, None)
        if not chunk:
            raise IOError("stream ended early")
        self._cpos += len(self._chunk)
        self._chunk = chunk
    def readinto(self, b):
        if self.pos >= self.size:
            return 0
        n = min(len(b), self.size - self.pos)
        for attempt in range(self.retries):
            try:
                end = self._cpos + len(self._chunk)
                if self._it is None or self.pos < self._cpos or self.pos > end + self.skip_ahead:
                    self._it, self._chunk, self._cpos = self.fetch(self.pos), b"", self.pos
                    self.reconnects += 1
                while self.pos >= self._cpos + len(self._chunk):
                    self._next_chunk()
                i = self.pos - self._cpos
                m = min(n, len(self._chunk) - i)
                b[:m] = self._chunk[i:i + m]
                self.pos += m
                return m
            except Exception as e:
                self._it = None
                if attempt == self.retries - 1:
                    raise
                wait = min(2 ** attempt, 60)
                print(f"  network hiccup at {self.pos / 1e9:.2f} GB ({type(e).__name__}); retrying in {wait}s")
                time.sleep(wait)

API = None
try:
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    auth.authenticate_user()
    creds, _ = google.auth.default()
    API = AuthorizedSession(creds)
    API.get("https://www.googleapis.com/drive/v3/about?fields=user", timeout=30).raise_for_status()
    print("Drive API: ok (zips will be streamed)")
except Exception as e:
    API = None
    print(f"Drive API unavailable ({type(e).__name__}: {e}); falling back to the Drive mount")

def open_zip_source(name):
    """Return (seekable binary file, size in bytes, how) for <name>.zip in the data folder."""
    if API is not None:
        q = f"name = '{name}.zip' and '{DATA_FOLDER_ID}' in parents and trashed = false"
        r = API.get("https://www.googleapis.com/drive/v3/files", timeout=60,
                    params={"q": q, "fields": "files(id,name,size,modifiedTime)", "orderBy": "modifiedTime desc"})
        r.raise_for_status()
        files = r.json().get("files", [])
        if not files:
            raise FileNotFoundError(f"{name}.zip not found in the Drive folder {DATA_FOLDER_ID}")
        fid, size = files[0]["id"], int(files[0]["size"])
        url = f"https://www.googleapis.com/drive/v3/files/{fid}?alt=media"
        def fetch(start):
            resp = API.get(url, headers={"Range": f"bytes={start}-"}, stream=True, timeout=120)
            resp.raise_for_status()
            if start and resp.status_code != 206:
                raise IOError(f"server ignored range request (HTTP {resp.status_code})")
            return resp.iter_content(8 << 20)
        return io.BufferedReader(RangeFile(size, fetch), 1 << 20), size, "Drive API stream"
    path = f"{DATA_FOLDER}/{name}.zip"
    return open(path, "rb"), os.path.getsize(path), "Drive mount"

def check_zip(name):
    """Read only the zip's table of contents (a few MB at the end of the file)."""
    try:
        f, size, how = open_zip_source(name)
        with zipfile.ZipFile(f) as zf:
            infos = [i for i in zf.infolist() if not i.is_dir()]
        f.close()
        return dict(ok=True, size=size, files=len(infos), unzipped=sum(i.file_size for i in infos), how=how)
    except Exception as e:
        return dict(ok=False, error=f"{type(e).__name__}: {e}")

Drive API: ok (zips will be streamed)


## 8. Check every zip before using it
Reads only each zip's table of contents, so it takes seconds. A zip that was cut off during download or upload shows **BAD** here; that subset is then skipped and you need to download and upload it again.

In [9]:
ZIP_STATUS = {}
for name in ["desdf"] + SUBSETS:
    s = check_zip(name)
    ZIP_STATUS[name] = s
    if s["ok"]:
        print(f"{name:9s} OK   zip {s['size'] / 1e9:6.2f} GB | {s['files']:7d} files | {s['unzipped'] / 1e9:6.2f} GB unzipped")
    else:
        print(f"{name:9s} BAD  {s['error']}")
assert ZIP_STATUS["desdf"]["ok"], "desdf.zip is required by every evaluation and could not be read."

desdf     OK   zip   0.03 GB |       9 files |   0.03 GB unzipped
gibson_t  OK   zip  16.43 GB |   51370 files |  16.42 GB unzipped
gibson_f  OK   zip  31.27 GB |   99593 files |  31.25 GB unzipped
gibson_g  OK   zip  61.32 GB |  198709 files |  61.28 GB unzipped


## 9. Unzip, sanity-check and provenance helpers

In [10]:
import yaml, hashlib, subprocess, datetime, platform, numpy, cv2, scipy, lightning, torchvision

DATA_ROOT = "/content/f3loc/data/Gibson Floorplan Localization Dataset"
WORK = "/content/data"
os.makedirs(DATA_ROOT, exist_ok=True)

def extract(name):
    dest = f"{WORK}/{name}"
    if os.path.exists(f"{dest}/.complete"):
        return dest
    shutil.rmtree(dest, ignore_errors=True)
    os.makedirs(dest)
    f, size, how = open_zip_source(name)
    with zipfile.ZipFile(f) as zf:
        infos = sorted((i for i in zf.infolist() if not i.is_dir()), key=lambda i: i.header_offset)
        need = sum(i.file_size for i in infos)
        free = shutil.disk_usage("/content").free
        assert need + 3e9 < free, (f"{name}: needs {need / 1e9:.1f} GB unzipped but only {free / 1e9:.1f} GB is free. "
                                   "Make sure DELETE_AFTER = True so earlier subsets are removed first.")
        print(f"unzipping {name} ({need / 1e9:.1f} GB, {len(infos)} files) via {how}")
        with tqdm(total=need, unit="B", unit_scale=True, desc=name) as bar:
            for i in infos:
                zf.extract(i, dest)
                bar.update(i.file_size)
    f.close()
    on_disk = sum(len(fs) for _, _, fs in os.walk(dest))
    assert on_disk == len(infos), f"{name}: {on_disk} files on disk but the zip lists {len(infos)}"
    open(f"{dest}/.complete", "w").write(f"{len(infos)} files\n")
    return dest

def link(name, dest):
    # locate the real folder inside the unzipped tree, whatever the zip's top-level layout
    if name == "desdf":
        hit = next(iter(glob.glob(f"{dest}/**/desdf.npy", recursive=True)), None)
        root = os.path.dirname(os.path.dirname(hit)) if hit else None
    else:
        hit = next(iter(glob.glob(f"{dest}/**/split.yaml", recursive=True)), None)
        root = os.path.dirname(hit) if hit else None
    assert root, f"{name}: could not find {'desdf.npy' if name == 'desdf' else 'split.yaml'} in {dest}"
    target = f"{DATA_ROOT}/{name}"
    if os.path.lexists(target):
        os.remove(target)
    os.symlink(root, target)
    return target

def unlink_and_delete(name):
    target = f"{DATA_ROOT}/{name}"
    if os.path.lexists(target):
        os.remove(target)
    shutil.rmtree(f"{WORK}/{name}", ignore_errors=True)

def summarize(ds):
    split = yaml.safe_load(open(f"{DATA_ROOT}/{ds}/split.yaml"))
    scenes = split["test"]
    lens = [sum(1 for _ in open(f"{DATA_ROOT}/{ds}/{s}/poses.txt")) for s in scenes]
    no_desdf = [s for s in scenes if not os.path.exists(f"{DATA_ROOT}/desdf/{s}/desdf.npy")]
    if ds == "gibson_t":
        unit, n = f"trajectories of {TRAJ_LEN}", sum(l // TRAJ_LEN for l in lens)
    else:
        unit, n = "4-frame samples", sum(l // 4 for l in lens)
    line = f"{ds}: {len(scenes)} test scenes, {sum(lens)} poses, {n} {unit}, missing DESDF: {no_desdf or 'none'}"
    assert not no_desdf, line
    return line

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for blk in iter(lambda: fh.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()

commit = subprocess.check_output(["git", "-C", "/content/f3loc", "rev-parse", "HEAD"], text=True).strip()
PROV = f"{RESULTS}/provenance.txt"
prov = [
    f"session_start_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')}",
    "platform: Google Colab",
    f"f3loc_commit: {commit}",
    f"python: {platform.python_version()}",
    f"torch: {torch.__version__} (cuda {torch.version.cuda})",
    f"torchvision: {torchvision.__version__}  lightning: {lightning.__version__}  numpy: {numpy.__version__}  scipy: {scipy.__version__}  opencv: {cv2.__version__}",
    f"gpu: {torch.cuda.get_device_name(0)}",
    *[f"ckpt {n}: sha256 {sha(f'/content/f3loc/logs/{n}')}" for n in names],
    *[f"zip {k}: {v['size']} bytes, {v['files']} files" if v["ok"] else f"zip {k}: BAD ({v['error']})" for k, v in ZIP_STATUS.items()],
]
with open(PROV, "a") as fh:
    fh.write("\n".join(prov) + "\n")
print("\n".join(prov))

def note(line):
    print(line)
    with open(PROV, "a") as fh:
        fh.write(line + "\n")

link("desdf", extract("desdf"))
print("desdf ready:", len(os.listdir(f"{DATA_ROOT}/desdf")), "scenes")

session_start_utc: 2026-10-25T14:57:18+00:00
platform: Google Colab
f3loc_commit: 9e8027d9219ca505078283ebfd925580f476ab97
python: 3.13.16
torch: 2.11.0+cu130 (cuda 13.0)
torchvision: 0.26.0+cu130  lightning: 2.5.6  numpy: 2.1.3  scipy: 1.16.3  opencv: 4.14.0
gpu: Tesla T4
ckpt mono.ckpt: sha256 950d1b9e6e9cd31532619767ed6c8ab9d4cb1870decf3d43eecfcf86bf8a86fb
ckpt mv.ckpt: sha256 793b3d2b77a208c9bfa13426b5914e9edc89fd31a61161f2f5365198d8cb0d26
ckpt comp.ckpt: sha256 5e5c6c764062b4a2ecda1accd8435639b6b070b5d62103b5e9345300a27b5246
zip desdf: 29364839 bytes, 9 files
zip gibson_t: 16429264687 bytes, 51370 files
zip gibson_f: 31274999793 bytes, 99593 files
zip gibson_g: 61321577890 bytes, 198709 files
unzipping desdf (0.0 GB, 9 files) via Drive API stream


desdf:   0%|          | 0.00/29.4M [00:00<?, ?B/s]

desdf ready: 9 scenes


## 10. Run the evaluations, one subset at a time
Per subset: unzip → count test scenes/frames (recorded in `provenance.txt`; these are the N behind every number) → run its evaluations → delete the unzipped files.

- `gibson_t`: sequential histogram filter, Table 3 (`d` = Ours_s, `comp` = Ours_f)
- `gibson_f`: single-frame, Table 1 (`d`, `mvd` = Ours_m; `comp_s`, `comp` run too but are not in the paper)
- `gibson_g`: single-frame, Table 2 (`d`, `mvd`, `comp_s` = Ours_t, `comp` = Ours_f)

In [11]:
import shlex

def run(tag, args, done_file):
    if os.path.exists(done_file):
        print("skip (already done):", tag)
        return True
    log = f"{RESULTS}/{tag}.log"
    cmd = f"cd /content/f3loc && python {args} 2>&1 | tee {shlex.quote(log)}"
    get_ipython().system(f"bash -o pipefail -c {shlex.quote(cmd)}")
    ok = os.path.exists(done_file)
    print(("DONE   " if ok else "FAILED ") + tag + ("" if ok else f"  (see {log})"))
    return ok

def jobs(ds):
    if ds == "gibson_t":
        return [(f"filt_{net}",
                 f"eval_filtering.py --net_type {net} --traj_len {TRAJ_LEN} --out_prefix {shlex.quote(f'{RESULTS}/filt_{net}')}",
                 f"{RESULTS}/filt_{net}_traj.csv") for net in ("comp", "d")]
    return [(f"obs_{ds}_{net}",
             f"eval_observation.py --net_type {net} --dataset {ds} --out_csv {shlex.quote(f'{RESULTS}/obs_{ds}_{net}.csv')}",
             f"{RESULTS}/obs_{ds}_{net}.csv") for net in ("comp", "d", "mvd", "comp_s")]

for ds in SUBSETS:
    todo = [j for j in jobs(ds) if not os.path.exists(j[2])]
    print(f"\n===== {ds}: {len(todo)} of {len(jobs(ds))} evaluations to run =====")
    if not todo:
        continue
    if not ZIP_STATUS[ds]["ok"]:
        print(f"SKIPPING {ds}: its zip is unreadable ({ZIP_STATUS[ds]['error']}). Re-download and re-upload {ds}.zip.")
        continue
    link(ds, extract(ds))
    note(summarize(ds))
    for tag, args, done in todo:
        run(tag, args, done)
    if DELETE_AFTER:
        unlink_and_delete(ds)
        print(f"deleted unzipped {ds}; free disk now {shutil.disk_usage('/content').free / 1e9:.0f} GB")


===== gibson_t: 0 of 2 evaluations to run =====

===== gibson_f: 0 of 4 evaluations to run =====

===== gibson_g: 4 of 4 evaluations to run =====
unzipping gibson_g (61.3 GB, 198709 files) via Drive API stream


gibson_g:   0%|          | 0.00/61.3G [00:00<?, ?B/s]

gibson_g: 9 test scenes, 16056 poses, 4014 4-frame samples, missing DESDF: none
/content/f3loc/modules/mono/depth_net.py:62: SyntaxWarning: invalid escape sequence '\='
  | \=============================> K_proj (30, 40, 32) ======/  |
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
======= USING DEVICE :  cuda  =======
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/chec

## 11. Results vs. paper
Verdict per the research plan: |delta| ≤ 1 pt OK, 1–2 pts CHECK, > 2 pts STOP and find out why. E1 is deterministic, so any non-zero delta is systematic, not sampling noise. Rows for subsets not yet run are simply absent.

In [12]:
from IPython.display import Markdown, display
!python /content/compare_e1.py "{RESULTS}" > "{RESULTS}/E1_table.md"
display(Markdown(open(f"{RESULTS}/E1_table.md").read()))

# E1 results: F3Loc released checkpoints vs. paper

```
session_start_utc: 2026-10-24T22:27:22+00:00
platform: Google Colab
f3loc_commit: 9e8027d9219ca505078283ebfd925580f476ab97
python: 3.13.16
torch: 2.11.0+cu130 (cuda 13.0)
torchvision: 0.26.0+cu130  lightning: 2.5.6  numpy: 2.1.3  scipy: 1.16.3  opencv: 4.14.0
gpu: Tesla T4
ckpt mono.ckpt: sha256 950d1b9e6e9cd31532619767ed6c8ab9d4cb1870decf3d43eecfcf86bf8a86fb
ckpt mv.ckpt: sha256 793b3d2b77a208c9bfa13426b5914e9edc89fd31a61161f2f5365198d8cb0d26
ckpt comp.ckpt: sha256 5e5c6c764062b4a2ecda1accd8435639b6b070b5d62103b5e9345300a27b5246
zip desdf: 29364839 bytes, 9 files
zip gibson_t: 16429264687 bytes, 51370 files
zip gibson_f: 31274999793 bytes, 99593 files
zip gibson_g: BAD (BadZipFile: File is not a zip file)
gibson_t: 9 test scenes, 4061 poses, 37 trajectories of 100, missing DESDF: none
gibson_f: 9 test scenes, 8028 poses, 2007 4-frame samples, missing DESDF: none
session_start_utc: 2026-10-25T14:57:18+00:00
platform: Google Colab
f3loc_commit: 9e8027d9219ca505078283ebfd925580f476ab97
python: 3.13.16
torch: 2.11.0+cu130 (cuda 13.0)
torchvision: 0.26.0+cu130  lightning: 2.5.6  numpy: 2.1.3  scipy: 1.16.3  opencv: 4.14.0
gpu: Tesla T4
ckpt mono.ckpt: sha256 950d1b9e6e9cd31532619767ed6c8ab9d4cb1870decf3d43eecfcf86bf8a86fb
ckpt mv.ckpt: sha256 793b3d2b77a208c9bfa13426b5914e9edc89fd31a61161f2f5365198d8cb0d26
ckpt comp.ckpt: sha256 5e5c6c764062b4a2ecda1accd8435639b6b070b5d62103b5e9345300a27b5246
zip desdf: 29364839 bytes, 9 files
zip gibson_t: 16429264687 bytes, 51370 files
zip gibson_f: 31274999793 bytes, 99593 files
zip gibson_g: 61321577890 bytes, 198709 files
gibson_g: 9 test scenes, 16056 poses, 4014 4-frame samples, missing DESDF: none
```

## Single-frame observation (Tables 1 and 2)

| dataset | net | paper row | N | metric | ours (%) | ±95% | paper (%) | delta | verdict |
|---|---|---|---|---|---|---|---|---|---|
| gibson_f | d | Ours_s | 2007 | R@0.1m | 4.7 | 0.9 | 4.7 | -0.0 | OK |
| gibson_f | d | Ours_s | 2007 | R@0.5m | 28.6 | 2.0 | 28.6 | -0.0 | OK |
| gibson_f | d | Ours_s | 2007 | R@1m | 36.6 | 2.1 | 36.6 | -0.0 | OK |
| gibson_f | d | Ours_s | 2007 | R@1m30° | 35.0 | 2.1 | 35.1 | -0.1 | OK |
| gibson_f | mvd | Ours_m | 2007 | R@0.1m | 13.3 | 1.5 | 13.2 | +0.1 | OK |
| gibson_f | mvd | Ours_m | 2007 | R@0.5m | 40.8 | 2.2 | 40.9 | -0.1 | OK |
| gibson_f | mvd | Ours_m | 2007 | R@1m | 45.2 | 2.2 | 45.2 | -0.0 | OK |
| gibson_f | mvd | Ours_m | 2007 | R@1m30° | 43.7 | 2.2 | 43.7 | -0.0 | OK |
| gibson_f | comp_s | Ours_t | 2007 | R@0.1m | 13.3 | 1.5 | not reported |  |  |
| gibson_f | comp_s | Ours_t | 2007 | R@0.5m | 40.8 | 2.2 | not reported |  |  |
| gibson_f | comp_s | Ours_t | 2007 | R@1m | 45.2 | 2.2 | not reported |  |  |
| gibson_f | comp_s | Ours_t | 2007 | R@1m30° | 43.7 | 2.2 | not reported |  |  |
| gibson_f | comp | Ours_f | 2007 | R@0.1m | 14.2 | 1.5 | not reported |  |  |
| gibson_f | comp | Ours_f | 2007 | R@0.5m | 42.1 | 2.2 | not reported |  |  |
| gibson_f | comp | Ours_f | 2007 | R@1m | 47.4 | 2.2 | not reported |  |  |
| gibson_f | comp | Ours_f | 2007 | R@1m30° | 45.6 | 2.2 | not reported |  |  |
| gibson_g | d | Ours_s | 4014 | R@0.1m | 4.3 | 0.6 | 4.3 | +0.0 | OK |
| gibson_g | d | Ours_s | 4014 | R@0.5m | 26.6 | 1.4 | 26.7 | -0.1 | OK |
| gibson_g | d | Ours_s | 4014 | R@1m | 33.6 | 1.5 | 33.7 | -0.1 | OK |
| gibson_g | d | Ours_s | 4014 | R@1m30° | 32.2 | 1.4 | 32.3 | -0.1 | OK |
| gibson_g | mvd | Ours_m | 4014 | R@0.1m | 9.3 | 0.9 | 9.3 | -0.0 | OK |
| gibson_g | mvd | Ours_m | 4014 | R@0.5m | 27.0 | 1.4 | 27.0 | -0.0 | OK |
| gibson_g | mvd | Ours_m | 4014 | R@1m | 30.9 | 1.4 | 31.0 | -0.1 | OK |
| gibson_g | mvd | Ours_m | 4014 | R@1m30° | 29.2 | 1.4 | 29.2 | -0.0 | OK |
| gibson_g | comp_s | Ours_t | 4014 | R@0.1m | 10.5 | 0.9 | 10.5 | -0.0 | OK |
| gibson_g | comp_s | Ours_t | 4014 | R@0.5m | 34.2 | 1.5 | 34.3 | -0.1 | OK |
| gibson_g | comp_s | Ours_t | 4014 | R@1m | 39.6 | 1.5 | 39.6 | -0.0 | OK |
| gibson_g | comp_s | Ours_t | 4014 | R@1m30° | 38.0 | 1.5 | 38.0 | +0.0 | OK |
| gibson_g | comp | Ours_f | 4014 | R@0.1m | 12.2 | 1.0 | 12.2 | -0.0 | OK |
| gibson_g | comp | Ours_f | 4014 | R@0.5m | 39.3 | 1.5 | 39.4 | -0.1 | OK |
| gibson_g | comp | Ours_f | 4014 | R@1m | 44.4 | 1.5 | 44.5 | -0.1 | OK |
| gibson_g | comp | Ours_f | 4014 | R@1m30° | 43.1 | 1.5 | 43.2 | -0.1 | OK |

## Sequential filtering on gibson_t, 100-frame history (Table 3)

| net | paper row | trajectories | success@1m ours (%) | paper | delta | RMSE succ ours / paper (m) | RMSE all ours / paper (m) | verdict |
|---|---|---|---|---|---|---|---|---|
| d | Ours_s | 37 | 89.2 | 89.2 | -0.0 | 0.18 / 0.18 | 0.88 / 0.88 | OK |
| comp | Ours_f | 37 | 94.6 | 94.6 | -0.0 | 0.12 / 0.12 | 0.51 / 0.51 | OK |

## Per-step timing (Table 4 context; ours on our GPU, paper hardware unstated)

Lab rule: distributions only. The paper's column is its reported mean, shown for orientation, not compared.

| net | stage | N steps | p50 (ms) | p99 (ms) | p99.9 (ms) | max (ms) | paper mean (ms) |
|---|---|---|---|---|---|---|---|
| d | feature | 3552 | 240.84 | 336.23 | 367.98 | 754.62 | n/a |
| d | matching | 3552 | 4.68 | 9.67 | 11.96 | 156.50 | n/a |
| d | iteration | 3552 | 248.25 | 342.69 | 378.72 | 945.36 | n/a |
| comp | feature | 3552 | 294.88 | 367.84 | 402.47 | 3257.11 | 33 |
| comp | matching | 3552 | 4.58 | 9.82 | 11.67 | 181.73 | 3 |
| comp | iteration | 3552 | 302.20 | 374.97 | 412.14 | 3458.43 | 37 |


## 12. Draft running-log entry
Copy into your log and edit the Suspect and Next lines by hand.

In [13]:
table = open(f"{RESULTS}/E1_table.md").read()
counts = {v: table.count(f"| {v} |") for v in ("OK", "CHECK", "STOP")}
done = sorted(os.path.basename(p)[:-4] for p in glob.glob(f"{RESULTS}/*.csv") if not p.endswith("_steps.csv"))
bad = [k for k, v in ZIP_STATUS.items() if not v["ok"]]
print(f'''## {datetime.date.today().isoformat()} (A)
Experiment: E1
Tried: released F3Loc checkpoints on Gibson ({", ".join(SUBSETS)}), Colab {torch.cuda.get_device_name(0)}, f3loc {commit[:7]}, results in {RESULTS}
Result: {len(done)} evaluations finished ({", ".join(done)}); {counts['OK']} OK, {counts['CHECK']} CHECK, {counts['STOP']} STOP against paper Tables 1-3{"; unreadable zips: " + ", ".join(bad) if bad else ""}
Suspect: <fill in>
Next: <fill in>
Blocked: <yes/no>''')

## 2026-10-25 (A)
Experiment: E1
Tried: released F3Loc checkpoints on Gibson (gibson_t, gibson_f, gibson_g), Colab Tesla T4, f3loc 9e8027d, results in /content/drive/MyDrive/Thesis/E1/results/gibson_v1
Result: 10 evaluations finished (filt_comp_traj, filt_d_traj, obs_gibson_f_comp, obs_gibson_f_comp_s, obs_gibson_f_d, obs_gibson_f_mvd, obs_gibson_g_comp, obs_gibson_g_comp_s, obs_gibson_g_d, obs_gibson_g_mvd); 26 OK, 0 CHECK, 0 STOP against paper Tables 1-3
Suspect: <fill in>
Next: <fill in>
Blocked: <yes/no>
